# add_messages 메시지 리듀서

사람과 모델의 메시지를 객체 형식으로 누적하면서 기존 메시지를 어떻게 수정할까요?

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** HumanMessage(content=..., id='h1')는 역할·본문·식별자를 가진 메시지 객체를 만듭니다. add_messages는 상태의 기존 messages와 노드가 반환한 새 메시지를 받습니다. 새 ID면 끝에 추가하고 같은 ID면 해당 위치의 객체를 교체합니다. AIMessage.text는 content가 여러 블록일 때도 텍스트 부분을 읽기 위한 속성입니다.

**다른 상황에 적용.** 상담원이 사용자의 오타를 수정하거나 도구 결과가 늦게 도착해 기존 메시지를 보완할 때 같은 ID 교체 규칙을 사용할 수 있습니다. 반대로 새 대화 턴은 새 ID를 부여해야 이전 발화를 덮어쓰지 않습니다.

**비교와 경계.** operator.add는 두 목록을 그대로 이어 붙이므로 같은 ID를 이해하지 못합니다. add_messages는 메시지 ID와 역할을 아는 전용 리듀서입니다. 다만 메시지를 상태에 저장한다고 장기 기억이 자동으로 생기는 것은 아니며, 체크포인터와 thread_id는 뒤 장에서 별도로 다룹니다.


## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 메시지 상태와 응답 노드 정의

다음 코드에서 메시지 상태와 응답 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `ChatState` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `reply(state: ChatState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `Annotated[value_type, reducer]`는 값의 형식과 새 갱신을 합칠 함수를 함께 표시합니다. `AIMessage(content=..., id=...)`는 모델 역할의 메시지 객체를 만듭니다.


In [2]:
# 메시지 상태와 응답 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing import Annotated
from typing_extensions import TypedDict
from langchain.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def reply(state: ChatState) -> dict:
    question = state['messages'][-1].content
    return {'messages': [AIMessage(content=f'확인한 질문: {question}', id='a1')]}

## 메시지 그래프 실행

메시지 그래프 실행 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `StateGraph(StateType)`은 상태 형식을 입력으로 받아 그래프 설계 객체를 만듭니다. `add_node(name, function)`은 그래프 이름과 실제 실행할 함수를 연결하며, 이 호출만으로 함수 본문이 실행되지는 않습니다. `add_edge(from, to)`는 두 노드의 고정 이동 순서를 등록합니다. `compile()`은 선언한 노드와 엣지로 `invoke()`할 실행 객체를 반환합니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다.


In [3]:
# 메시지 그래프 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder = StateGraph(ChatState)
builder.add_node('reply', reply)
builder.add_edge(START, 'reply')
builder.add_edge('reply', END)
graph = builder.compile()
first = graph.invoke({'messages': [HumanMessage(content='대출 기간?', id='h1')]})
for message in first['messages']:
    print(message.type, message.id, message.content)

human h1 대출 기간?
ai a1 확인한 질문: 대출 기간?


## AIMessage.text 확인

AIMessage.text 확인 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# AIMessage.text 확인 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
answer_message = first['messages'][-1]
print('AIMessage.text:', answer_message.text)
assert answer_message.text == '확인한 질문: 대출 기간?'

AIMessage.text: 확인한 질문: 대출 기간?


## 같은 ID 메시지 교체 비교

같은 ID 메시지 교체 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

**이 셀의 코드 읽기.** `add_messages(left, right)`는 메시지 ID를 기준으로 새 메시지를 추가하거나 같은 ID의 메시지를 교체한 목록을 반환합니다. `HumanMessage(content=..., id=...)`는 사용자 역할의 본문과 식별자를 가진 메시지 객체를 만듭니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
revised = add_messages(
    first['messages'],
    [HumanMessage(content='도서 대출 기간은?', id='h1')],
)
for message in revised:
    print(message.type, message.id, message.content)
assert len(revised) == 2
assert revised[0].content == '도서 대출 기간은?'

human h1 도서 대출 기간은?
ai a1 확인한 질문: 대출 기간?
